# RoadVision AI — Vast.ai GPU notebook

Your original workflow remains in this notebook. Paths and GPU setup are adapted for Vast.ai and the project folders. Run cells in order. Training uses the selected CUDA GPU.


## Vast.ai setup
Use a PyTorch/CUDA Jupyter template, preferably with Python 3.10 or 3.11. The selected image supplies PyTorch; the project requirements do not pin a CUDA build. Restart the kernel and rerun setup if package installation requests it.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess

# Launch Jupyter from the repository root or notebooks/.
current = Path.cwd().resolve()
ROOT = next((p for p in [current, *current.parents]
             if (p / "notebooks/roadvision.ipynb").is_file()
             and (p / "requirements.txt").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the RoadVision-AI- repository.")

DATA_DIR = ROOT / "data"
MODELS_DIR = ROOT / "models"
REPORTS_DIR = ROOT / "reports"
YOLO_DIR = ROOT / "third_party" / "yolov9"
for folder in [DATA_DIR, MODELS_DIR, REPORTS_DIR, YOLO_DIR.parent]:
    folder.mkdir(parents=True, exist_ok=True)

os.environ["KAGGLEHUB_CACHE"] = str(DATA_DIR / "kaggle_cache")
clean_dataset = DATA_DIR / "road_vehicle_clean"
SPLIT_ROOT = DATA_DIR / "road_vehicle_split"
weights_path = MODELS_DIR / "yolov9-c.pt"
TEST_FRACTION = 0.15
EPOCHS = 10
BATCH_SIZE = 4
IMAGE_SIZE = 640
print("Project:", ROOT)


In [ ]:
# Install into this notebook's Python environment.
subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                str(ROOT / "requirements.txt")], check=True)


In [ ]:
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select a Vast.ai PyTorch/CUDA GPU template and a GPU notebook kernel.")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


## 1. Imports


In [ ]:
from pathlib import Path
from collections import Counter
import shutil
import random

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import yaml




## 2. Download the dataset


In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "ashfakyeafi/road-vehicle-images-dataset"
)

print("Dataset downloaded to:", dataset_path)



## 3. Inspect the dataset folder


In [ ]:
import os

print("Dataset folder:")
print(dataset_path)

print("\nFiles and folders:")
for item in os.listdir(dataset_path):
    print(item)



## 4. Count images and annotations


In [ ]:
from pathlib import Path

dataset = Path(dataset_path)

images = [
    p for p in dataset.rglob("*")
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
]

labels = list(dataset.rglob("*.txt"))
yaml_files = list(dataset.rglob("*.yaml"))

print("Images:", len(images))
print("Annotations:", len(labels))
print("YAML files:", len(yaml_files))



## 5. Preview the first image


In [ ]:
if images:
    image = Image.open(images[0])

    plt.figure(figsize=(8, 6))
    plt.imshow(image)
    plt.axis("off")
    plt.title("First image from our dataset")
    plt.show()
else:
    print("No images found!")



## 6. Show the dataset tree


In [ ]:
def show_tree(folder, max_depth=3):
    folder = Path(folder)

    for path in sorted(folder.rglob("*")):
        depth = len(path.relative_to(folder).parts)

        if depth <= max_depth:
            indent = "  " * (depth - 1)
            print(indent + path.name)

show_tree(dataset, max_depth=2)



## 7. Load the dataset configuration


In [ ]:
yaml_files = sorted(dataset.rglob("data_1.yaml"))

if not yaml_files:
    raise FileNotFoundError(
        "No data.yaml found. Check the dataset folder."
    )

print("Found configuration files:")
for file in yaml_files:
    print(file)

YAML_PATH = yaml_files[0]

with open(YAML_PATH, "r") as f:
    config = yaml.safe_load(f)

print("\nDataset configuration:")
print(config)



## 8. Read class names


In [ ]:
names = config["names"]

if isinstance(names, dict):
    class_names = {
        int(k): str(v) for k, v in names.items()
    }
else:
    class_names = {
        i: str(name) for i, name in enumerate(names)
    }

num_classes = len(class_names)

print("\nNumber of classes:", num_classes)
display("Classes:", class_names)

assert set(class_names) == set(range(num_classes)), \
    "Class IDs must be consecutive, starting at 0."



## 9. Collect image files


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

image_files = sorted([
    p for p in dataset.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
])

print("Total images:", len(image_files))

print("\nFirst five images:")
for image in image_files[:5]:
    print(image.relative_to(dataset))



## 10. Match images with labels


In [ ]:
def find_label(image_path):
    parts = list(image_path.parts)

    image_positions = [
        i for i, part in enumerate(parts)
        if part.lower() == "images"
    ]

    if not image_positions:
        raise ValueError(
            f"Cannot locate 'images' folder for {image_path}"
        )

    # Replace the images directory with labels.
    idx = image_positions[-1]
    parts[idx] = "labels"

    return Path(*parts).with_suffix(".txt")


label_paths = [find_label(p) for p in image_files]

print("Images:", len(image_files))
print(
    "Existing matching labels:",
    sum(p.exists() for p in label_paths)
)
print(
    "Images missing labels:",
    sum(not p.exists() for p in label_paths)
)



## 11. Inspect an annotation


In [ ]:
existing_labels = [
    p for p in label_paths if p.is_file()
]

if existing_labels:
    example_label = existing_labels[0]

    print("Label file:", example_label.name)
    print("\nContents:")
    print(example_label.read_text())
else:
    print("No annotation files were found.")



## 12. Inspect annotation size and contents


In [ ]:
example_label = existing_labels[0]

print("File:", example_label.name)
print("File size:", example_label.stat().st_size, "bytes")
print("Content:", repr(example_label.read_text()))



## 13. Find non-empty annotations


In [ ]:
non_empty_labels = [
    p for p in existing_labels
    if p.read_text().strip()
]

print("Total label files:", len(existing_labels))
print("Non-empty labels:", len(non_empty_labels))
print(
    "Empty labels:",
    len(existing_labels) - len(non_empty_labels)
)

if non_empty_labels:
    example_label = non_empty_labels[0]

    print("\nLabel file:", example_label.name)
    print("\nContents:")
    print(example_label.read_text())
else:
    print("All annotation files are empty!")



## 14. List empty annotations


In [ ]:
empty_labels = [
    p for p in existing_labels
    if not p.read_text().strip()
]

print("Empty annotation files:")

for label in empty_labels:
    print(label)



## 15. Find images for empty annotations


In [ ]:
from pathlib import Path

dataset = Path(dataset_path)

for label_path in empty_labels:
    print("\nLooking for:", label_path.name)

    image_name = label_path.stem

    matches = [
        p for p in dataset.rglob("*")
        if p.is_file()
        and p.stem == image_name
        and p.suffix.lower() in {
            ".jpg", ".jpeg", ".png", ".webp", ".bmp"
        }
    ]

    if matches:
        for image in matches:
            print("✅ Image found:", image)
    else:
        print("❌ No matching image found")



## 16. Preview empty-label images in grayscale


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

for label_path in empty_labels:

    matches = [
        p for p in dataset.rglob("*")
        if p.is_file()
        and p.stem == label_path.stem
        and p.suffix.lower() in {
            ".jpg", ".jpeg", ".png", ".webp", ".bmp"
        }
    ]

    for image_path in matches:
        with Image.open(image_path) as img:
            plt.figure(figsize=(10, 6))
            plt.imshow(img.convert("L"))

        plt.title(image_path.name)
        plt.axis("off")
        plt.show()



## 17. Preview empty-label images in color


In [ ]:
for label_path in empty_labels:

    image_base = Path(
        *[
            "images" if part == "labels" else part
            for part in label_path.parts
        ]
    ).with_suffix("")

    found = False

    for extension in [".jpg", ".jpeg", ".png", ".webp"]:
        image_path = image_base.with_suffix(extension)

        if image_path.exists():
            image = Image.open(image_path)

            plt.figure(figsize=(9, 5))
            plt.imshow(image)
            plt.title(f"Empty annotation: {label_path.name}")
            plt.axis("off")
            plt.show()

            found = True
            break

    if not found:
        print("Image not found for:", label_path.name)



## 18. Check unmatched filenames


In [ ]:
image_stems = {p.stem for p in images}
label_stems = {p.stem for p in existing_labels}

print("Images with no matching label:",
      len(image_stems - label_stems))

print("Labels with no matching image:",
      len(label_stems - image_stems))

print("\nExample unmatched labels:")
print(list(label_stems - image_stems)[:10])



## 19. Copy the dataset for cleaning


In [ ]:
# Use a new directory for each preparation run to avoid stale files.
if clean_dataset.exists():
    raise FileExistsError(f"{clean_dataset} already exists. Choose a new clean_dataset path in setup.")

shutil.copytree(
    dataset,
    clean_dataset,
    dirs_exist_ok=False
)

print("Clean dataset created:", clean_dataset)



## 20. Remove empty-label pairs from the copy


In [ ]:
for label_path in empty_labels:

    clean_label = clean_dataset / label_path.relative_to(dataset)

    matching_images = [
        p for p in image_files
        if p.stem == label_path.stem
        and p.parent.parent.name == label_path.parent.parent.name
    ]

    if len(matching_images) == 1:
        clean_image = (
            clean_dataset / matching_images[0].relative_to(dataset)
        )

        clean_image.unlink()
        clean_label.unlink()

        print("Removed:", clean_image.name)
        print("Removed:", clean_label.name)

    else:
        print("Skipped:", label_path.name)



## 21. Count remaining files


In [ ]:
clean_images = [
    p for p in clean_dataset.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
]

clean_labels = list(clean_dataset.rglob("*.txt"))

print("Images remaining:", len(clean_images))
print("Labels remaining:", len(clean_labels))



## 22. Validate class IDs


In [ ]:
invalid_classes = []

for label_path in clean_labels:

    for line_number, line in enumerate(
        label_path.read_text().splitlines(), start=1
    ):
        values = line.split()

        if not values:
            continue

        try:
            class_id = int(values[0])

            if class_id not in class_names:
                invalid_classes.append(
                    (label_path.name, line_number, class_id)
                )

        except ValueError:
            invalid_classes.append(
                (label_path.name, line_number, values[0])
            )

print("Invalid class IDs:", len(invalid_classes))

for error in invalid_classes[:10]:
    print(error)



## 23. Validate bounding boxes


In [ ]:
import math

invalid_boxes = []

for label_path in clean_labels:

    for line_number, line in enumerate(
        label_path.read_text().splitlines(), start=1
    ):
        values = line.split()

        try:
            if len(values) != 5:
                raise ValueError("Expected 5 values")

            class_value, x, y, w, h = map(float, values)

            if not all(math.isfinite(v) for v in
                       [class_value, x, y, w, h]):
                raise ValueError("Non-finite number")

            if not class_value.is_integer():
                raise ValueError("Invalid class ID format")

            if not (0 <= x <= 1 and 0 <= y <= 1):
                raise ValueError("Invalid center coordinates")

            if not (0 < w <= 1 and 0 < h <= 1):
                raise ValueError("Invalid box size")

            if (
                x - w/2 < -1e-6 or
                x + w/2 > 1 + 1e-6 or
                y - h/2 < -1e-6 or
                y + h/2 > 1 + 1e-6
            ):
                raise ValueError("Box outside image")

        except ValueError as error:
            invalid_boxes.append(
                (label_path.name, line_number, str(error))
            )

print("Invalid annotation lines:", len(invalid_boxes))

for error in invalid_boxes[:len(invalid_boxes)]:
    print(error)



## 24. Summarize invalid annotation files


In [ ]:
affected_files = Counter(
    filename for filename, line, error in invalid_boxes
)

print("Invalid annotation lines:", len(invalid_boxes))
print("Affected files:", len(affected_files))

for filename, count in affected_files.items():
    print(f"{filename}: {count} invalid lines")



## 25. Remove pairs with invalid boxes


In [ ]:
deleted = []

for image_path in clean_images:

    label_path = find_label(image_path)

    if label_path.name in affected_files:
        image_path.unlink()
        label_path.unlink()

        deleted.append(image_path.name)
        print("Deleted:", image_path.name)

print("\nTotal images deleted:", len(deleted))



## 26. Check remaining annotations


In [ ]:
clean_images = [p for p in clean_images if p.exists()]
clean_labels = [p for p in clean_labels if p.exists()]

print("Remaining images:", len(clean_images))
print("Remaining labels:", len(clean_labels))

remaining_errors = []

for filename, line_number, error in invalid_boxes:
    if any(p.name == filename for p in clean_labels):
        remaining_errors.append((filename, line_number, error))

print("Remaining previously detected errors:", len(remaining_errors))



## 27. Check image integrity


In [ ]:
corrupted_images = []

for image_path in clean_images:
    try:
        with Image.open(image_path) as img:
            img.verify()

    except Exception as error:
        corrupted_images.append((image_path, str(error)))

print("Total images checked:", len(clean_images))
print("Corrupted images:", len(corrupted_images))

for image_path, error in corrupted_images:
    print(image_path.name, ":", error)



## 28. Check image decoding


In [ ]:
unreadable_images = []

for image_path in clean_images:
    try:
        with Image.open(image_path) as img:
            img.load()

    except Exception as error:
        unreadable_images.append((image_path, str(error)))

print("Images that cannot be decoded:", len(unreadable_images))



## 29. Count objects per class


In [ ]:
class_counts = Counter()

for label_path in clean_labels:
    with open(label_path, "r") as f:
        for line in f:
            if line.strip():
                class_id = int(line.split()[0])
                class_counts[class_id] += 1

print("Class Distribution:\n")

for class_id, class_name in class_names.items():
    print(f"{class_name}: {class_counts[class_id]}")



## 30. Build the class distribution table


In [ ]:
class_distribution = pd.DataFrame([
    {
        "Class ID": class_id,
        "Class Name": class_name,
        "Count": class_counts[class_id]
    }
    for class_id, class_name in class_names.items()
])

class_distribution = class_distribution.sort_values(
    by="Count", ascending=False
)

display(class_distribution)



## 31. Plot class distribution


In [ ]:
plt.figure(figsize=(12, 6))

plt.bar(
    class_distribution["Class Name"],
    class_distribution["Count"],
    color="steelblue"
)

plt.title("RoadVision AI - Class Distribution")
plt.xlabel("Vehicle Classes")
plt.ylabel("Number of Objects")

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()



## 32. Collect training and validation images


In [ ]:
train_images = [
    p for p in clean_images
    if p.parent.parent.name == "train"
]

valid_images = [
    p for p in clean_images
    if p.parent.parent.name == "valid"
]

print("Train candidates:", len(train_images))
print("Validation:", len(valid_images))

if len(train_images) < 2 or not valid_images:
    raise ValueError("Need at least two training images and one validation image.")
train_candidate_count = len(train_images)
valid_candidate_count = len(valid_images)



## 33. Create the test split


In [ ]:
from sklearn.model_selection import train_test_split

train_images, test_images = train_test_split(
    sorted(train_images),
    test_size=max(1, min(train_candidate_count - 1, round(train_candidate_count * TEST_FRACTION))),
    random_state=42
)

print("Train:", len(train_images))
print("Validation:", len(valid_images))
print("Test:", len(test_images))



## 34. Create split directories


In [ ]:
if SPLIT_ROOT.exists():
    raise FileExistsError(f"{SPLIT_ROOT} already exists. Choose a new SPLIT_ROOT in setup.")

for split in ["train", "valid", "test"]:
    (SPLIT_ROOT / "images" / split).mkdir(
        parents=True, exist_ok=True
    )
    (SPLIT_ROOT / "labels" / split).mkdir(
        parents=True, exist_ok=True
    )

print("Dataset folders created!")



## 35. Copy image-label pairs into splits


In [ ]:
splits = {
    "train": train_images,
    "valid": valid_images,
    "test": test_images
}

for split_name, image_list in splits.items():

    for image_path in image_list:
        label_path = find_label(image_path)

        if not label_path.is_file():
            raise FileNotFoundError(label_path)

        shutil.copy2(
            image_path,
            SPLIT_ROOT / "images" / split_name / image_path.name
        )

        shutil.copy2(
            label_path,
            SPLIT_ROOT / "labels" / split_name / label_path.name
        )

    print(split_name, ":", len(image_list))



## 36. Verify split counts


In [ ]:
for split in ["train", "valid", "test"]:

    image_count = len(list(
        (SPLIT_ROOT / "images" / split).iterdir()
    ))

    label_count = len(list(
        (SPLIT_ROOT / "labels" / split).glob("*.txt")
    ))

    print(
        f"{split}: {image_count} images, "
        f"{label_count} labels"
    )

    assert image_count == label_count

assert len(train_images) + len(test_images) == train_candidate_count
assert len(valid_images) == valid_candidate_count
assert set(train_images).isdisjoint(test_images)

print("Dataset split verified!")



## 37. Clone YOLOv9


In [ ]:
# Clone before using the checkout; no dependence on the current directory.
if not YOLO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/WongKinYiu/yolov9.git",
                    str(YOLO_DIR)], check=True)
print("YOLOv9:", YOLO_DIR)


## 38. Check YOLOv9 files


In [ ]:
from pathlib import Path

repo = YOLO_DIR

files_to_check = [
    "train_dual.py",
    "models/detect/yolov9-c.yaml",
    "data/hyps/hyp.scratch-high.yaml",
    "requirements.txt"
]

for file in files_to_check:
    print(file, ":", (repo / file).exists())





## 39. Install YOLOv9 dependencies


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                str(YOLO_DIR / "requirements.txt")], check=True)
# Restart the kernel if the installer requests it. Prepared data can be reused
# by rerunning setup and the configuration/YOLOv9 cells, skipping preparation.


## 40. Check GPU availability


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



## 41. Write the training configuration


In [ ]:
from pathlib import Path
import yaml

data_config = {
    "train": str(SPLIT_ROOT / "images/train"),
    "val": str(SPLIT_ROOT / "images/valid"),
    "test": str(SPLIT_ROOT / "images/test"),
    "nc": len(class_names),
    "names": [class_names[i] for i in range(len(class_names))]
}

DATA_YAML = SPLIT_ROOT / "data.yaml"

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

print(DATA_YAML.read_text())



## 42. Download pretrained weights


In [ ]:
from urllib.request import urlretrieve

if not weights_path.is_file():
    temporary = weights_path.with_suffix(".download")
    urlretrieve(
        "https://github.com/WongKinYiu/yolov9/releases/download/v0.1/yolov9-c.pt",
        temporary,
    )
    temporary.replace(weights_path)
print("Pretrained weights:", weights_path)


## 43. Check weights and runtime


In [ ]:
import torch

weights_path = MODELS_DIR / "yolov9-c.pt"

print("Weights exist:", weights_path.is_file())

if weights_path.is_file():
    print("Weights size (MB):",
          round(weights_path.stat().st_size / 1024**2, 2))

print("Dataset config:", DATA_YAML.is_file())
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



## 44. Disable Weights & Biases


In [ ]:
import os

os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"




## 45. Train YOLOv9


In [ ]:
from datetime import datetime, timezone

# Check CUDA again in a fresh process after installing YOLOv9 dependencies.
subprocess.run([sys.executable, "-c",
    "import torch; assert torch.cuda.is_available(), 'CUDA GPU unavailable'; "
    "print(torch.cuda.get_device_name(0))"], check=True)
if not DATA_YAML.is_file() or not weights_path.is_file():
    raise FileNotFoundError("Prepare data.yaml and download pretrained weights first.")

RUN_NAME = "roadvision_yolov9c_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
result_dir = REPORTS_DIR / RUN_NAME
# Keep a record outside the run directory so YOLOv9 can create that directory.
import json
run_record = {
    "run_name": RUN_NAME, "data_yaml": str(DATA_YAML.resolve()),
    "epochs": EPOCHS, "batch_size": BATCH_SIZE, "image_size": IMAGE_SIZE,
    "architecture": "yolov9-c", "training_script": "train_dual.py",
}
(REPORTS_DIR / f"{RUN_NAME}.json").write_text(json.dumps(run_record, indent=2))
subprocess.run([
    sys.executable, "train_dual.py",
    "--weights", str(weights_path),
    "--cfg", "models/detect/yolov9-c.yaml",
    "--data", str(DATA_YAML),
    "--hyp", "data/hyps/hyp.scratch-high.yaml",
    "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
    "--img", str(IMAGE_SIZE), "--device", "0", "--workers", "2",
    "--project", str(REPORTS_DIR), "--name", RUN_NAME,
], cwd=YOLO_DIR, check=True)
print("Results:", result_dir)
print("Best checkpoint:", result_dir / "weights" / "best.pt")
for filename in ["best.pt", "last.pt"]:
    checkpoint = result_dir / "weights" / filename
    if not checkpoint.is_file() or checkpoint.stat().st_size == 0:
        raise RuntimeError(f"Training finished without a usable checkpoint: {checkpoint}")
print("Verified: best.pt and last.pt were saved by YOLOv9.")


## 46. Select and verify the saved training run
YOLOv9 saves `best.pt` and `last.pt` automatically in the training run's `weights/` folder. `best.pt` is selected using validation performance. `last.pt` is the final saved checkpoint, not a guarantee of resumable optimizer state after upstream checkpoint stripping.

After a kernel restart, run the Vast.ai setup cell and this cell. Set `RUN_TO_USE` to the run folder name when more than one run exists. No retraining or dataset cleaning is required to export an existing run.


In [ ]:
from pathlib import Path
import json
import yaml

RUN_TO_USE = ""  # Example: roadvision_yolov9c_20261107_120000_123456
available = sorted(p.parent.parent for p in REPORTS_DIR.glob("*/weights/best.pt"))
print("Available runs:", [p.name for p in available])
if RUN_TO_USE:
    result_dir = REPORTS_DIR / RUN_TO_USE
elif "result_dir" not in globals():
    if len(available) != 1:
        raise ValueError("Set RUN_TO_USE to one of the printed run names.")
    result_dir = available[0]
result_dir = Path(result_dir).resolve()
if result_dir.parent != REPORTS_DIR.resolve():
    raise ValueError("Select a run directly inside reports/.")
BEST_WEIGHTS = result_dir / "weights" / "best.pt"
LAST_WEIGHTS = result_dir / "weights" / "last.pt"
for checkpoint in [BEST_WEIGHTS, LAST_WEIGHTS]:
    if not checkpoint.is_file() or checkpoint.stat().st_size == 0:
        raise FileNotFoundError(f"Missing or empty checkpoint: {checkpoint}")
    print(checkpoint.name, round(checkpoint.stat().st_size / 1024**2, 2), "MiB")

record_path = REPORTS_DIR / f"{result_dir.name}.json"
run_record = json.loads(record_path.read_text()) if record_path.is_file() else {}
# For older runs, read the options saved by YOLOv9.
options_path = result_dir / "opt.yaml"
options = yaml.safe_load(options_path.read_text()) if options_path.is_file() else {}
data_location = run_record.get("data_yaml") or options.get("data")
if not data_location:
    raise ValueError("Cannot identify this run's dataset. Set its data_yaml in reports/<run_name>.json.")
DATA_YAML = Path(data_location)
if not DATA_YAML.is_absolute():
    DATA_YAML = (YOLO_DIR / DATA_YAML).resolve()
if not DATA_YAML.is_file():
    raise FileNotFoundError(f"Restore this run's dataset configuration: {DATA_YAML}")
selected_config = yaml.safe_load(DATA_YAML.read_text())
raw_names = selected_config["names"]
class_names = ({int(k): str(v) for k, v in raw_names.items()}
               if isinstance(raw_names, dict) else dict(enumerate(raw_names)))
if not class_names or set(class_names) != set(range(len(class_names))):
    raise ValueError("Class IDs must start at zero and be consecutive.")
print("Selected run:", result_dir.name)
print("Classes:", class_names)


## 47. Show training results


In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

# result_dir is set by the training cell. After a kernel restart, set it
# explicitly to REPORTS_DIR / "your_saved_run_name".

if (result_dir / "results.png").exists():
    img = Image.open(result_dir / "results.png")

    plt.figure(figsize=(16, 10))
    plt.imshow(img)
    plt.axis("off")
    plt.show()
else:
    print("Training results not found yet.")



## 48. Find result plots


In [ ]:
from pathlib import Path

runs_dir = REPORTS_DIR

result_images = list(runs_dir.rglob("results.png"))

print("Results images found:", len(result_images))

for path in result_images:
    print(path)



## 49. Generate plots from result CSV files


In [ ]:
# Run upstream plotting in a subprocess to avoid changing notebook imports/cwd.
csv_files = sorted(REPORTS_DIR.rglob("results.csv"))
print("CSV files found:", len(csv_files))
for csv_file in csv_files:
    print("Processing:", csv_file)
    subprocess.run([
        sys.executable, "-c",
        "import sys; from pathlib import Path; from utils.plots import plot_results; "
        "plot_results(file=Path(sys.argv[1]))", str(csv_file),
    ], cwd=YOLO_DIR, check=True)


## 50. Save the model and its metadata
Create a versioned bundle under `models/<run_name>/`. Keep both saved checkpoints, class names, dataset configuration, training options, and environment details. The app uses the stable copy `models/best.pt`; rerunning this cell selects this run for the app. The original run remains in `reports/`.


In [ ]:
import hashlib
import shutil
import subprocess
import sys
from datetime import datetime, timezone

BUNDLE_DIR = MODELS_DIR / result_dir.name
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

for source in [BEST_WEIGHTS, LAST_WEIGHTS]:
    destination = BUNDLE_DIR / source.name
    shutil.copy2(source, destination)
    if sha256_file(source) != sha256_file(destination):
        raise RuntimeError(f"Checkpoint copy verification failed: {destination}")
shutil.copy2(DATA_YAML, BUNDLE_DIR / "training_data.yaml")
for filename in ["opt.yaml", "hyp.yaml", "results.csv", "results.png"]:
    source = result_dir / filename
    if source.is_file():
        shutil.copy2(source, BUNDLE_DIR / filename)
(BUNDLE_DIR / "classes.json").write_text(json.dumps(class_names, indent=2))
revision = subprocess.run(["git", "rev-parse", "HEAD"], cwd=YOLO_DIR,
                          capture_output=True, text=True, check=True).stdout.strip()
packages = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                         capture_output=True, text=True, check=True).stdout
(BUNDLE_DIR / "environment.txt").write_text(packages)
metadata = {
    "run_name": result_dir.name, "architecture": "yolov9-c",
    "inference_script": "detect_dual.py", "yolov9_commit": revision,
    "exported_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version, "class_names": class_names,
    "training": run_record or options,
    "sha256": {p.name: sha256_file(p) for p in [BEST_WEIGHTS, LAST_WEIGHTS]},
    "test_evaluation": "not run by this export",
    "prediction_check": "not run by this export",
}
(BUNDLE_DIR / "model_info.json").write_text(json.dumps(metadata, indent=2, default=str))
# Publish the verified best checkpoint without leaving a partial best.pt.
app_weights = MODELS_DIR / "best.pt"
temporary = MODELS_DIR / "best.pt.tmp"
shutil.copy2(BUNDLE_DIR / "best.pt", temporary)
temporary.replace(app_weights)
print("Saved model bundle:", BUNDLE_DIR)
print("Streamlit checkpoint:", app_weights)


## 51. Evaluate the exported model on the held-out test split
Run this once the model is selected. This is a separate test evaluation, not the training/validation curves. It saves the console metrics and plots. Do not tune the model repeatedly against the test results.


In [ ]:
from datetime import datetime, timezone

if not (YOLO_DIR / "val_dual.py").is_file():
    raise FileNotFoundError("Run the YOLOv9 setup cells first.")
EVAL_NAME = "test_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
EVAL_DIR = result_dir / EVAL_NAME
command = [
    sys.executable, "val_dual.py", "--weights", str(BUNDLE_DIR / "best.pt"),
    "--data", str(DATA_YAML), "--task", "test", "--imgsz", str(IMAGE_SIZE),
    "--batch-size", str(BATCH_SIZE), "--device", "0", "--workers", "2",
    "--project", str(result_dir), "--name", EVAL_NAME, "--verbose",
]
# Keep the full evaluation log even if evaluation fails.
with (BUNDLE_DIR / "test_metrics.log").open("w") as log:
    evaluation = subprocess.run(command, cwd=YOLO_DIR, stdout=log,
                                stderr=subprocess.STDOUT, text=True)
print((BUNDLE_DIR / "test_metrics.log").read_text()[-12000:])
if evaluation.returncode:
    raise RuntimeError("Test evaluation failed; inspect test_metrics.log.")
for source in EVAL_DIR.glob("*.png"):
    shutil.copy2(source, BUNDLE_DIR / ("test_" + source.name))
metadata["test_evaluation"] = {"status": "completed", "output": str(EVAL_DIR)}
(BUNDLE_DIR / "model_info.json").write_text(json.dumps(metadata, indent=2, default=str))
print("Test evaluation saved:", EVAL_DIR)


## 52. Verify prediction using the same code as Streamlit
This reloads the exported checkpoint through the app's inference helper and displays one test image with detections. A successful export alone does not prove the model can run in the current environment.


In [ ]:
from PIL import Image
from IPython.display import display
import io

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.inference import detect_image

# Set SAMPLE_IMAGE to a particular image if desired.
SAMPLE_IMAGE = None
if SAMPLE_IMAGE is None:
    test_images_dir = Path(selected_config["test"])
    if not test_images_dir.is_absolute():
        data_base = Path(selected_config.get("path", DATA_YAML.parent))
        if not data_base.is_absolute():
            data_base = DATA_YAML.parent / data_base
        test_images_dir = data_base / test_images_dir
    samples = sorted(p for p in test_images_dir.rglob("*")
                     if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"})
    if not samples:
        raise FileNotFoundError("No test image found. Set SAMPLE_IMAGE explicitly.")
    SAMPLE_IMAGE = samples[0]
annotated, detections = detect_image(
    Path(SAMPLE_IMAGE).read_bytes(), BUNDLE_DIR / "best.pt", YOLO_DIR,
    confidence=0.25, device="0",
)
(BUNDLE_DIR / "sample_prediction.jpg").write_bytes(annotated)
(BUNDLE_DIR / "sample_detections.txt").write_text(detections)
display(Image.open(io.BytesIO(annotated)))
print("Detected objects:", len(detections.splitlines()))
metadata["prediction_check"] = "passed using src.inference.detect_image"
(BUNDLE_DIR / "model_info.json").write_text(json.dumps(metadata, indent=2, default=str))


## 53. Package and download your saved model
The ZIP contains the checkpoints and export metadata, plus evaluation and prediction outputs if those cells were run. Download it from the Jupyter file browser before destroying the Vast.ai instance. The ZIP does not include the dataset, Python environment, or YOLOv9 source code.


In [ ]:
import zipfile

archive = MODELS_DIR / f"{result_dir.name}_model.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_STORED) as zipped:
    for source in sorted(BUNDLE_DIR.iterdir()):
        if source.is_file():
            zipped.write(source, arcname=source.name)
with zipfile.ZipFile(archive) as zipped:
    if zipped.testzip() is not None:
        raise RuntimeError("The model archive failed its integrity check.")
print("Download from the Jupyter file browser:", archive.relative_to(ROOT))
print("Archive size (MiB):", round(archive.stat().st_size / 1024**2, 2))


## 54. Install and launch Streamlit
Run this installation cell, then paste the printed launch command into a terminal. On another machine, extract the model ZIP, copy `best.pt` into this repository's `models/` folder, and install the recorded YOLOv9 version and compatible dependencies. The archived dataset paths refer to the training machine; update them only if you plan to evaluate there.


In [ ]:
import shlex

subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                str(ROOT / "requirements-streamlit.txt")], check=True)
print("Run in a terminal using the same Python environment:")
print(shlex.join([sys.executable, "-m", "streamlit", "run", str(ROOT / "app.py"),
                  "--server.address", "0.0.0.0", "--server.port", "8501"]))
print("Open the Vast.ai mapped external port for container port 8501.")
